# SUMO Trip Metrics Comparison Notebook

This Jupyter Notebook loads two SUMO tripinfo XML files—one using SUMO’s default settings and the other optimized with a multi-agent system—and compares multiple key metrics to demonstrate the performance improvements of the multi-agent approach.

---


## 1. Environment Setup


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt

## 2. Function to Load tripinfo Data with Extended Metrics


In [ ]:
def load_tripinfo(file_path, scenario_label):
    """Parse a SUMO tripinfo XML and return a DataFrame with extended metrics."""
    tree = ET.parse(file_path)
    root = tree.getroot()
    rows = []
    for trip in root.findall('tripinfo'):
        a = trip.attrib
        rows.append({
            'scenario': scenario_label,
            'id': a['id'],
            'depart': float(a['depart']),
            'arrival': float(a['arrival']),
            'duration': float(a['duration']),
            'waitingTime': float(a['waitingTime']),
            'timeLoss': float(a.get('timeLoss', 0.0)),
            'waitingCount': int(a.get('waitingCount', 0)),
        })
    return pd.DataFrame(rows)

## 3. Load Data and Combine Scenarios


In [ ]:
# File paths
file_default = 'trips_default.xml'
file_multi = 'trips_multi.xml'

# Load DataFrames with labels
df_default = load_tripinfo(file_default, 'default')
df_multi = load_tripinfo(file_multi, 'multi_agent')
# Concatenate into a single DataFrame
df = pd.concat([df_default, df_multi], ignore_index=True)
df.head()

## 4. Descriptive Statistics for All Metrics


In [ ]:
metrics = ['duration', 'waitingTime', 'timeLoss','waitingCount']
stats = df.groupby('scenario')[metrics].agg(['mean', 'median', 'std', 'min', 'max'])
print(stats)

## 5. Visual Comparison with Boxplots


In [ ]:
for metric in metrics:
    plt.figure(figsize=(6,4))
    df.boxplot(column=metric, by='scenario')
    plt.title(f'{metric.title()} by Scenario')
    plt.suptitle('')  # Remove automatic title
    plt.xlabel('Scenario')
    plt.ylabel(metric)
    plt.show()

## 6. Compute Percent Improvement of Multi-Agent over Default


In [ ]:
# Calculate mean values and improvement percentages
mean_values = df.groupby('scenario')[metrics].mean().T
mean_values['improvement_pct'] = (
    (mean_values['default'] - mean_values['multi_agent']) / mean_values['default'] * 100
)
print(mean_values)

## 8. Analysis and Conclusions

Based on the computed metrics, the multi-agent system shows substantial performance gains over the default scenario:

* **Trip Duration**: Reduced from an average of 102.29 s to 58.51 s, a **42.79%** improvement in travel time.
* **Waiting Time**: Decreased from an average of 55.24 s to 14.40 s, a **73.94%** reduction in intersection delays.
* **Time Loss**: Lowered from 74.68 s to 32.29 s, achieving a **56.77%** decrease in time lost compared to ideal travel.
* **Waiting Count**: Dropped from 2.26 stops per trip to 1.70 stops, representing a **24.75%** fewer stopping events.

These results clearly demonstrate that the multi-agent optimization significantly enhances traffic flow by shortening trips, reducing stops, and minimizing delays without increasing route distances.